In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    """Locate the FAME repository root from the current working directory."""
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "results").exists()
        ):
            return candidate
    raise RuntimeError(
        "FAME repository root not found. Run this notebook from inside a clone "
        "of the FAME repository."
    )

REPO_ROOT = find_repo_root()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
REPRODUCED_DIR = REPO_ROOT / "reproduced"
REPRODUCED_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "H1": REPRODUCED_DIR / "fantasy_football" / "H1",
    "H2": REPRODUCED_DIR / "fantasy_football" / "H2",
    "PRIMARY": REPRODUCED_DIR / "fantasy_football" / "primary",
}

OUTPUT_DIR = REPRODUCED_DIR / "fantasy_football" / "comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

META = {
    "H1": {
        "development":"2021",
        "calibration":2022,
        "test":2023,
        "role":"exploratory",
    },
    "H2": {
        "development":"2021-2022",
        "calibration":2023,
        "test":2024,
        "role":"historical replication",
    },
    "PRIMARY": {
        "development":"2021-2023",
        "calibration":2024,
        "test":2025,
        "role":"primary frozen experiment",
    },
}

for k, path in PATHS.items():
    print(k, path.resolve(), "exists=", path.exists())


In [ ]:
def find_existing(root, candidates):
    for rel in candidates:
        p = root / rel
        if p.exists():
            return p
    return None

def read_first(root, candidates):
    p = find_existing(root, candidates)
    if p is None:
        raise FileNotFoundError(
            f"None of the expected files were found under {root}: {candidates}"
        )
    return pd.read_csv(p), p

records = []

for key, root in PATHS.items():
    test_year = META[key]["test"]

    doc, doc_path = read_first(
        root,
        [
            "03_calibracao_decisao/frozen_doc_weights.csv",
        ],
    )

    op, op_path = read_first(
        root,
        [
            f"04_resultados_{test_year}/operational_summary_{test_year}.csv",
            "04_resultados_2025/operational_summary_2025.csv",
        ],
    )

    inf, inf_path = read_first(
        root,
        [
            f"04_resultados_{test_year}/primary_doc_vs_expected_normalized_inference.csv",
            "04_resultados_2025/primary_doc_vs_expected_normalized_inference.csv",
        ],
    )

    post, post_path = read_first(
        root,
        [
            f"04_resultados_{test_year}/posthoc_{test_year}_weight_landscape.csv",
            "04_resultados_2025/posthoc_2025_weight_landscape.csv",
        ],
    )

    stab, stab_path = read_first(
        root,
        [
            "03_calibracao_decisao/doc_calibration_bootstrap_stability.csv",
        ],
    )

    wdoc = doc.iloc[0]
    bestpost = post.sort_values("rank_mean").iloc[0]
    primary = inf.iloc[0]

    doc_mean = float(
        op.loc[op["strategy"].eq("FAME-DOC"), "mean_utility"].iloc[0]
    )
    exp_mean = float(
        op.loc[
            op["strategy"].eq("Expected-only normalized representation"),
            "mean_utility"
        ].iloc[0]
    )

    mask = (
        np.isclose(stab["w_expected"], float(wdoc["w_expected"]))
        & np.isclose(stab["w_upside"], float(wdoc["w_upside"]))
        & np.isclose(stab["w_economic"], float(wdoc["w_economic"]))
    )
    freq = float(stab.loc[mask, "selection_frequency_pct"].iloc[0]) if mask.any() else np.nan

    wcal = np.array([
        float(wdoc["w_expected"]),
        float(wdoc["w_upside"]),
        float(wdoc["w_economic"]),
    ])
    wpost = np.array([
        float(bestpost["w_expected"]),
        float(bestpost["w_upside"]),
        float(bestpost["w_economic"]),
    ])

    records.append({
        "experiment": key,
        **META[key],
        "w_doc_expected": wcal[0],
        "w_doc_dispersion": wcal[1],
        "w_doc_economic": wcal[2],
        "doc_bootstrap_selection_pct": freq,
        "doc_test_mean": doc_mean,
        "expected_norm_test_mean": exp_mean,
        "delta_doc_minus_expected_norm": doc_mean-exp_mean,
        "paired_ci95_low": float(primary["ci95_low"]),
        "paired_ci95_high": float(primary["ci95_high"]),
        "posthoc_best_expected": wpost[0],
        "posthoc_best_dispersion": wpost[1],
        "posthoc_best_economic": wpost[2],
        "l1_doc_to_posthoc": float(np.abs(wcal-wpost).sum()),
        "l2_doc_to_posthoc": float(np.sqrt(((wcal-wpost)**2).sum())),
    })

comparison = pd.DataFrame(records)
comparison


In [ ]:
comparison.to_csv(
    OUTPUT_DIR / "fame_temporal_replications_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

display(comparison)


In [ ]:
plt.figure(figsize=(8,5.2))

order = comparison.sort_values("test")
y = np.arange(len(order))
x = order["delta_doc_minus_expected_norm"].to_numpy(float)
lo = order["paired_ci95_low"].to_numpy(float)
hi = order["paired_ci95_high"].to_numpy(float)

plt.errorbar(
    x,
    y,
    xerr=np.vstack([x-lo, hi-x]),
    fmt="o",
    capsize=4
)
plt.axvline(0, linestyle="--", linewidth=1)
plt.yticks(
    y,
    [
        f'{r.experiment}: {r.calibration}→{r.test}'
        for r in order.itertuples()
    ]
)
plt.xlabel("Mean utility difference: FAME-DOC − Expected-only normalized")
plt.ylabel("")
plt.title("Temporal replication of the primary operational contrast")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "fig_temporal_replications_primary_contrast.pdf",
    bbox_inches="tight"
)
plt.savefig(
    OUTPUT_DIR / "fig_temporal_replications_primary_contrast.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
weight_cols = [
    "experiment","calibration","test",
    "w_doc_expected","w_doc_dispersion","w_doc_economic",
    "posthoc_best_expected","posthoc_best_dispersion","posthoc_best_economic",
    "l1_doc_to_posthoc","l2_doc_to_posthoc",
    "doc_bootstrap_selection_pct",
]
display(comparison[weight_cols])
